# 🧠 K-Fold Cross-Validation from Scratch

ยินดีต้อนรับสู่สมุดโน้ตอธิบายการใช้งานจริงสำหรับ **Cross-Validation**! ในสมุดโน้ตเล่มนี้ เราจะ:
1. อธิบายกลไกการทำงานของ $K$-Fold Cross-Validation
2. อิมพลีเมนต์ **ตัวแบ่งข้อมูล K-Fold จากศูนย์ (K-Fold Splitter from scratch)** โดยใช้การจัดแบ่งดัชนี (index partitioning) ด้วย NumPy
3. ฝึกฝนตัวจำแนกประเภทแบบง่าย (เช่น `DecisionTreeClassifier`) โดยใช้การตรวจสอบแบบ $K$-Fold เพื่อคำนวณความแม่นยำในแต่ละกลุ่มย่อย (fold accuracies) ค่าเฉลี่ย และส่วนเบี่ยงเบนมาตรฐาน
4. ตรวจสอบความถูกต้องของการแบ่งข้อมูลของฟังก์ชันที่เราเขียนขึ้นเอง เทียบกับคลาส `KFold` ของ `scikit-learn`
5. อภิปรายแนวคิด **Stratified K-Fold** สำหรับชุดข้อมูลที่คลาสไม่สมดุล
6. สรุปวิธีการนำ $K$-Fold ไปใช้ในกระบวนการเรียนรู้เชิงลึก (Deep Learning) และการทำโมเดลรวมกลุ่ม (Ensembling) ของ YOLO

เรามาเริ่มด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างข้อมูลจำลอง (Simulating a Dataset)

เราจะสร้างชุดข้อมูล 2 มิติ (2D dataset) แบบง่ายที่มีกลุ่มตัวอย่าง 150 ตัวอย่าง และมีทั้งหมด 2 คลาสครับ

In [ ]:
# Generate synthetic dataset
X = np.random.rand(150, 2)
y = (X[:, 0] > 0.5).astype(int)
noise_idx = np.random.choice(150, 15, replace=False)
y[noise_idx] = 1 - y[noise_idx]

## 2. การอิมพลีเมนต์ K-Fold Cross-Validation จากศูนย์ (from Scratch)

เรามาเขียนคลาสสำหรับแบ่งข้อมูล `CustomKFold` กันครับ:
-   `__init__(self, n_splits=5, shuffle=True, random_state=None)`
-   `split(self, X)`:
    -   สลับ (shuffle) อาร์เรย์ดัชนีข้อมูล
    -   แบ่งดัชนีออกเป็นอาร์เรย์กลุ่มย่อยๆ จำนวน `n_splits` กลุ่ม (folds)
    -   คืนค่า (yield) คู่ดัชนี `(train_idx, val_idx)` สำหรับแต่ละรอบ (fold)

In [ ]:
class CustomKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state

    def split(self, X):
        m = X.shape[0]
        indices = np.arange(m)
        
        if self.shuffle:
            if self.random_state is not None:
                np.random.seed(self.random_state)
            np.random.shuffle(indices)
            
        # Determine sizes of folds
        fold_sizes = np.full(self.n_splits, m // self.n_splits)
        fold_sizes[:m % self.n_splits] += 1
        
        current = 0
        folds = []
        for size in fold_sizes:
            folds.append(indices[current:current + size])
            current += size
            
        for i in range(self.n_splits):
            val_idx = folds[i]
            train_idx = np.concatenate([folds[j] for j in range(self.n_splits) if j != i])
            yield train_idx, val_idx

# Verify splitter fold sizes
custom_kf = CustomKFold(n_splits=5, shuffle=True, random_state=42)
for idx, (train, val) in enumerate(custom_kf.split(X)):
    print(f"Fold {idx+1} | Train Size: {len(train)} | Val Size: {len(val)}")

## 3. วงรอบการฝึกฝนและการประเมินผล (Training and Evaluation Loop)

เรามารันวงรอบการทดสอบหาความถูกต้องแบบ $K$-fold กันครับ:
-   ในแต่ละรอบ (fold) ฝึกฝนโมเดล `DecisionTreeClassifier` ด้วยดัชนีสำหรับฝึกฝน (training indices) ของรอบนั้นๆ
-   ประเมินค่าความแม่นยำ (accuracy) ด้วยดัชนีสำหรับตรวจสอบ (validation indices) ของรอบนั้นๆ
-   คำนวณค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานของความแม่นยำจากการทดสอบในทุกรอบ

In [ ]:
fold_accuracies = []

for idx, (train_idx, val_idx) in enumerate(custom_kf.split(X)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    clf = DecisionTreeClassifier(max_depth=3, random_state=42)
    clf.fit(X_train, y_train)
    
    preds = clf.predict(X_val)
    acc = accuracy_score(y_val, preds)
    fold_accuracies.append(acc)
    print(f"Fold {idx+1} Accuracy: {acc * 100:.2f}%")

mean_acc = np.mean(fold_accuracies)
std_acc = np.std(fold_accuracies)

print(f"\nOverall K-Fold CV Accuracy: {mean_acc * 100:.2f}% (± {std_acc * 100:.2f}%)")

## 4. การแสดงผลความแม่นยำในแต่ละกลุ่มย่อย (Visualizing Fold Accuracies)

เรามาพล็อตเพื่อแสดงให้เห็นภาพความเสถียรของผลการทำนายจากโมเดลในแต่ละรอบ (folds) กันครับ

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), [acc * 100 for acc in fold_accuracies], color='skyblue', edgecolor='black')
plt.axhline(mean_acc * 100, color='red', linestyle='--', label=f'Mean Accuracy: {mean_acc*100:.2f}%')
plt.xlabel('Fold Number')
plt.ylabel('Validation Accuracy (%)')
plt.title('K-Fold Cross Validation Accuracies')
plt.ylim(0, 100)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 💡 Stratified K-Fold (การรักษาความสมดุลของคลาส)
หากข้อมูล Class 1 มีสัดส่วนเพียงแค่ $2\%$ ของชุดข้อมูลทั้งหมด การใช้ K-Fold แบบสุ่มทั่วไปอาจทำให้บางรอบของการแบ่งข้อมูลมีจำนวน Class 1 เป็น 0 ตัวอย่างเลยก็เป็นได้ครับ
เพื่อแก้ปัญหานี้ **Stratified K-Fold** จะแยกข้อมูลกลุ่ม positive และ negative ออกจากกันอย่างเป็นอิสระ แล้วแบ่งแต่ละกลุ่มเป็น K ส่วนเท่าๆ กัน จากนั้นค่อยนำมาจับคู่รวมกันใหม่ ซึ่งวิธีนี้จะช่วยรับประกันว่าข้อมูลในทุกรอบยังคงรักษาสัดส่วนของ Class 1 ไว้ที่ $2\%$ ได้อย่างแม่นยำครับ

## 💡 ความเชื่อมโยงกับกระบวนการเรียนรู้เชิงลึกและ YOLO
ในการเรียนรู้เชิงลึก (Deep Learning) ยุคปัจจุบัน การฝึกฝนโมเดลเพียงโมเดลเดียว (เช่น ฝึก YOLO บนชุดข้อมูล COCO) อาจต้องใช้เวลาหลายวัน การทำ 5-Fold cross-validation จะทำให้ทรัพยากรและเวลาที่ต้องใช้คูณเพิ่มเป็น 5 เท่า ซึ่งมักจะไม่สามารถทำได้ในทางปฏิบัติครับ
อย่างไรก็ตาม K-Fold ยังคงได้รับความนิยมอย่างมากในสถานการณ์ทางคอมพิวเตอร์วิชันต่อไปนี้:
1.  **ชุดข้อมูลขนาดเล็กที่สร้างขึ้นเอง (Small Custom Datasets):** หากคุณมีรูปภาพที่ทำขึ้นเองเพียง 200 ภาพ การใช้ K-Fold จะช่วยป้องกันอคติ (bias) ที่อาจเกิดขึ้นจากการแบ่งชุดข้อมูลฝึกฝน/ชุดข้อมูลตรวจสอบเพียงครั้งเดียวได้ครับ
2.  **การอนุมานแบบรวมโมเดล (Ensemble Inference หรือ K-Fold Blending):** คุณฝึกฝนโมเดล 5 โมเดล (แต่ละโมเดลฝึกด้วยข้อมูลจากการแบ่งแบบแต่ละ fold) ในเวลาทดสอบ (test time) คุณจะส่งภาพอินพุตผ่านโมเดลทั้ง 5 โมเดลนี้ แล้วหาค่าเฉลี่ยผลการทำนายกล่องขอบเขต (bounding box predictions) ของทุกโมเดลเข้าด้วยกัน (โดยใช้เทคนิค Non-Maximum Suppression) วิธีนี้จะช่วยเพิ่มความแข็งแกร่ง (robustness) และความแม่นยำของโมเดลได้อย่างมหาศาลครับ